In [1]:
# De ser necesario descargar las siguientes librerias 
# !pip install sodapy
# !pip install requests 
# !pip install pyarrow

# Conexión a la API REST (SODA API)
import requests  
# Cliente oficial de Socrata 
from sodapy import Socrata 

import os  
import pandas as pd
import os
import psutil
print("Librerias Cargadas")

Librerias Cargadas


In [2]:
dataset_id = "p6dx-8zbt"
url = f"https://www.datos.gov.co/resource/{dataset_id}.json"

# Contar el TOTAL de registros
params = {"$select": "count(*)"}
res = requests.get(url, params=params)
total_registros = int(res.json()[0]['count'])
print(f" TOTAL de registros en la base: {total_registros:,}")

# Para saber cuántas variables tiene
meta_url = f"https://www.datos.gov.co/api/views/{dataset_id}.json"
meta_res = requests.get(meta_url).json()

# Extraer columnas declaradas en el metadata
vars_info = pd.DataFrame([{
    "Nombre": col['name'],
    "Campo (API)": col['fieldName'],
    "Tipo (API)": col['dataTypeName']
} for col in meta_res['columns']])

total_variables = len(vars_info)
print(f" TOTAL de variables en la base: {total_variables}")
# Vista de las variables y sus tipo por si se desea saber
# print("\n Variables disponibles:")
# print(vars_info)

# Ultima fecha registrada en la API
params_fecha = {
    "$select": "max(fecha_de_publicacion_del)"
}
res_fecha = requests.get(url, params=params_fecha)
ultima_fecha = res_fecha.json()[0]['max_fecha_de_publicacion_del']

print(f" Última fecha registrada en la API: {ultima_fecha}")


 TOTAL de registros en la base: 7,827,129
 TOTAL de variables en la base: 59
 Última fecha registrada en la API: 2025-10-05T00:00:00.000


In [6]:
# Ruta local en Google Drive (ajústala si tu unidad tiene otro nombre)
ruta = r"G:\Mi unidad\DatosAPI_Trimestral_Parquet"
salida = os.path.join(ruta, "Historico_Procesos.parquet")

# Listar archivos parquet (ordenados por nombre, que debería corresponder al orden cronológico)
archivos = sorted([f for f in os.listdir(ruta) if f.endswith(".parquet") and f != "Historico_Procesos.parquet"])

In [7]:
# Función para convertir columnas de fechas
def transformar_fechas(df):
    """Convierte todas las columnas que tengan 'fecha' o 'date' en datetime64[ns, UTC]"""
    for col in df.columns:
        if "fecha" in col.lower() or "date" in col.lower():
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
            except Exception:
                pass
    return df

# Crear histórico vacío si no existe
if os.path.exists(salida):
    historico = pd.read_parquet(salida)
    ultimo_procesado = historico["fecha_de_publicacion_del"].max()
    print(f" Histórico cargado. Última fecha procesada: {ultimo_procesado}")
else:
    historico = pd.DataFrame()
    ultimo_procesado = None
    historico.to_parquet(salida, index=False)
    print(" Creado nuevo archivo histórico vacío.")

# === Procesar archivo por archivo ===
for archivo in archivos:
    print(f"\n📄 Procesando archivo: {archivo}")
    df = pd.read_parquet(os.path.join(ruta, archivo))
    df = transformar_fechas(df)

    # 🔹 Eliminar columnas que ya no existen en la API
    for col in ["anio", "trimestre"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
           # print(f" Columna '{col}' eliminada del archivo {archivo}")

    # Si ya hay histórico, no volver a cargar lo procesado
    if ultimo_procesado is not None:
        if df["fecha_de_publicacion_del"].max() <= ultimo_procesado:
            print(f" {archivo} ya estaba incluido en el histórico, se omite.")
            continue

    # Concatenar y guardar
    historico = pd.concat([historico, df], axis=0, join="outer", ignore_index=True)
    print(f"✅ Trimestre {archivo} agregado al histórico.")

# === Guardado final ===
historico.to_parquet(salida, index=False)
print(f"\n Histórico consolidado y guardado en: {salida}")




 Creado nuevo archivo histórico vacío.

📄 Procesando archivo: datos_trimestre_2015_Q2_2015-04-01_to_2015-06-30.parquet
✅ Trimestre datos_trimestre_2015_Q2_2015-04-01_to_2015-06-30.parquet agregado al histórico.

📄 Procesando archivo: datos_trimestre_2015_Q3_2015-07-01_to_2015-09-30.parquet
✅ Trimestre datos_trimestre_2015_Q3_2015-07-01_to_2015-09-30.parquet agregado al histórico.

📄 Procesando archivo: datos_trimestre_2015_Q4_2015-10-01_to_2015-12-31.parquet
✅ Trimestre datos_trimestre_2015_Q4_2015-10-01_to_2015-12-31.parquet agregado al histórico.

📄 Procesando archivo: datos_trimestre_2016_Q1_2016-01-01_to_2016-03-31.parquet
✅ Trimestre datos_trimestre_2016_Q1_2016-01-01_to_2016-03-31.parquet agregado al histórico.

📄 Procesando archivo: datos_trimestre_2016_Q2_2016-04-01_to_2016-06-30.parquet
✅ Trimestre datos_trimestre_2016_Q2_2016-04-01_to_2016-06-30.parquet agregado al histórico.

📄 Procesando archivo: datos_trimestre_2016_Q3_2016-07-01_to_2016-09-30.parquet
✅ Trimestre datos_tri

In [ ]:
# Cargar histórico
historico = pd.read_parquet(salida)

# Número de filas y columnas
print("📊 Dimensiones del histórico:", historico.shape)
print(f"➡️ Total de registros: {len(historico):,}")

In [ ]:
import os
import shutil
notebook_actual = "Historico_de_Datos.ipynb"
ruta_destino = r"G:\Mi unidad\Colab Notebooks"

os.makedirs(ruta_destino, exist_ok=True)
shutil.copy(notebook_actual, os.path.join(ruta_destino, notebook_actual))

print(f"✅ Notebook guardado en {ruta_destino}")
